### Research Question 5

**Does the model know when it is wrong?**

In [5]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [6]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [7]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run xraymodel.ipynb

In [8]:
results = []

for pathology in DATASET_PATHOLOGIES:

    label_idx = DATASET_LABEL_INDICES_MAP[pathology]

    y_true = XRAY_LABELS_LIST[:, label_idx]
    y_score = MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx]

    y_pred = (y_score >= 0.7).astype(int)

    fp = (y_true == 0) & (y_pred == 1)

    fp_scores = y_score[fp]

    if len(fp_scores) == 0:
        continue

    results.append({
        "pathology": pathology,
        "n_false_positives": len(fp_scores),
        "mean_fp_score": fp_scores.mean(),
        "median_fp_score": np.median(fp_scores),
        "max_fp_score": fp_scores.max(),
        "fp_ge_0.9": np.mean(fp_scores >= 0.9),
        "fp_ge_0.95": np.mean(fp_scores >= 0.95)
    })

fp_results = pd.DataFrame(results)

fp_results.sort_values(
    "fp_ge_0.9",
    ascending=False
)

,pathology,n_false_positives,mean_fp_score,median_fp_score,max_fp_score,fp_ge_0.9,fp_ge_0.95
7,Lung Opacity,1732,0.850209,0.858788,0.998144,0.324480,0.117783
8,Effusion,1539,0.849360,0.857896,0.996427,0.320338,0.108512
1,Cardiomegaly,1952,0.838411,0.838038,0.997130,0.288422,0.125000
0,Atelectasis,3098,0.840030,0.843650,0.995429,0.243060,0.059716
6,Lung Lesion,1142,0.811838,0.801431,0.999108,0.161996,0.052539
3,Edema,1757,0.807886,0.800296,0.985427,0.138304,0.022766
2,Consolidation,1606,0.807636,0.799767,0.993711,0.124533,0.021793
4,Enlarged Cardiomediastinum,1362,0.796878,0.784617,0.990635,0.094714,0.019090
5,Fracture,871,0.790735,0.774854,0.993846,0.089552,0.021814
9,Pneumonia,1175,0.782775,0.769696,0.988604,0.051915,0.010213


In [9]:
THRESHOLD = 0.7

def prepare_pathology_data(pathology, threshold=THRESHOLD):

    label_idx = DATASET_LABEL_INDICES_MAP[pathology]

    # Original ground-truth labels and model scores
    y_true_raw = np.asarray(XRAY_LABELS_LIST[:, label_idx])
    y_score_raw = np.asarray(
        MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx]
    )

    # Only use explicitly known binary labels
    # Important: don't treat unknown labels as negatives.
    valid = np.isin(y_true_raw, [0, 1])

    y_true = y_true_raw[valid].astype(int)
    y_score = y_score_raw[valid]

    y_pred = (y_score >= threshold).astype(int)

    # Classification categories
    tp = (y_true == 1) & (y_pred == 1)
    tn = (y_true == 0) & (y_pred == 0)
    fp = (y_true == 0) & (y_pred == 1)
    fn = (y_true == 1) & (y_pred == 0)

    # Confidence in the predicted class
    prediction_confidence = np.where(
        y_pred == 1,
        y_score,
        1 - y_score
    )

    df = pd.DataFrame({
        "ground_truth": y_true,
        "prediction": y_pred,
        "score": y_score,
        "prediction_confidence": prediction_confidence,
        "outcome": np.select(
            [tp, tn, fp, fn],
            ["True Positive", "True Negative",
             "False Positive", "False Negative"], default=None
        ),
        "age": np.asarray(XRAY_AGE_LIST)[valid],
        "sex": np.asarray(XRAY_GENDER_LIST)[valid],
        "view": np.asarray(XRAY_VIEW_LIST)[valid]
    })

    return df

In [10]:
pathology = "Pneumonia"

df = prepare_pathology_data(pathology)

df.head()

,ground_truth,prediction,score,prediction_confidence,outcome,age,sex,view
0,0,0,0.429010,0.570990,True Negative,68,1.0,ap
1,0,1,0.705091,0.705091,False Positive,87,1.0,ap
2,0,0,0.126731,0.873269,True Negative,41,0.0,ap
3,0,0,0.399492,0.600508,True Negative,20,1.0,pa
4,0,0,0.522499,0.477501,True Negative,33,0.0,pa


In [11]:
fig = px.histogram(
    df,
    x="prediction_confidence",
    color="outcome",
    barmode="overlay",
    histnorm="probability density",
    nbins=40,
    opacity=0.65,
    color_discrete_map={
        "True Positive": "#009E73",
        "True Negative": "#0072B2",
        "False Positive": "#D55E00",
        "False Negative": "#CC79A7"
    },
    hover_data=["score", "age", "sex", "view"],
    title=f"Does the Model Know When It Is Wrong? – {pathology}"
)

fig.add_vline(
    x=0.9,
    line_dash="dash",
    line_color="black",
    annotation_text="High confidence (0.9)"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Confidence in predicted class",
    yaxis_title="Probability density",
    legend_title="Prediction outcome",
    hovermode="x unified",
    width=1000,
    height=600
)

fig.show()

In [12]:
fp_df = df[df["outcome"] == "False Positive"].copy()

fig = px.histogram(
    fp_df,
    x="score",
    nbins=40,
    color_discrete_sequence=["#E74C3C"],
    marginal="box",
    title=f"Confidence Distribution of False Positives – {pathology}",
    hover_data=["age", "sex", "view"]
)

fig.add_vline(
    x=0.7,
    line_dash="dash",
    line_color="black",
    annotation_text="Decision threshold (0.7)"
)

fig.add_vline(
    x=0.9,
    line_dash="dot",
    line_color="blue",
    annotation_text="High confidence (0.9)"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Model score",
    yaxis_title="Number of false positives",
    width=1000,
    height=600
)

fig.show()

In [13]:
bins = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.000001]

labels = [
    "0.70–0.75",
    "0.75–0.80",
    "0.80–0.85",
    "0.85–0.90",
    "0.90–0.95",
    "0.95–1.00"
]

fp_df["confidence_range"] = pd.cut(
    fp_df["score"],
    bins=bins,
    labels=labels,
    right=False
)

fp_counts = (
    fp_df["confidence_range"]
    .value_counts(sort=False)
    .rename_axis("Confidence range")
    .reset_index(name="Number of FPs")
)

fp_counts["Percentage"] = (
    fp_counts["Number of FPs"] /
    max(len(fp_df), 1) * 100
)

fig = px.bar(
    fp_counts,
    x="Confidence range",
    y="Number of FPs",
    text="Percentage",
    color="Number of FPs",
    color_continuous_scale="Reds",
    title=f"False Positives by Confidence Range – {pathology}",
    hover_data={"Percentage": ":.2f"}
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Model score",
    yaxis_title="Number of false positives",
    width=950,
    height=550
)

fig.show()